# 1. Setup

In [ ]:
import sys
sys.path.append("/home/565/pv3484/aus_substation_electricity")
%cd /home/565/pv3484/aus_substation_electricity
!pwd

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import matplotlib.patches as mpatches


# 2. Load data

In [ ]:
demand_rank = pd.read_csv("/home/565/pv3484/aus_substation_electricity/data/cleaned_data/full_nsw_relative_rank.csv")
weather = pd.read_csv("/home/565/pv3484/aus_substation_electricity/data/BOM_NSW_weather_processed_v3/weather_station_066062.csv")

print("demand_rank shape:", demand_rank.shape)
print("weather shape:", weather.shape)

# 3. Filter Mosman and build paired dataframe

In [ ]:
station_code = 'MOSMA'
station_full_name = 'Mosman'

# Filter Mosman demand rank
mosman_rank = demand_rank[
    demand_rank['station_code'].str.contains('MOS', case=False, na=False)
].copy()

rank_cols = ['date', 'holiday_group', 'is_weekend', 'is_holiday'] + \
            [c for c in mosman_rank.columns if 'rank' in c.lower()]
mosman_rank = mosman_rank[rank_cols]
mosman_rank['date'] = pd.to_datetime(mosman_rank['date']).dt.normalize()

# Aggregate weather into time blocks (2004-2017)
weather['timestamp_utc'] = pd.to_datetime(weather['timestamp_utc'], utc=True)
weather = weather[
    (weather['timestamp_utc'].dt.year >= 2004) &
    (weather['timestamp_utc'].dt.year <= 2017)
]
weather['timestamp_local'] = weather['timestamp_utc'].dt.tz_localize(None)
weather['date'] = weather['timestamp_local'].dt.normalize()
weather['time_block'] = weather['timestamp_local'].dt.hour.map(lambda h:
    '00_04' if h < 4 else '04_10' if h < 10 else '10_15' if h < 15 else '15_20' if h < 20 else '20_24'
)

met_cols = ['temp', 'dewpoint', 'relative_humidity']
weather_agg = weather.groupby(['date', 'time_block'])[met_cols].mean().reset_index()

weather_wide = weather_agg.pivot(index='date', columns='time_block', values=met_cols)
weather_wide.columns = [f"{var}_{block}" for var, block in weather_wide.columns]
weather_wide = weather_wide.reset_index()
weather_wide['date'] = pd.to_datetime(weather_wide['date']).dt.normalize()

# Merge
paired = mosman_rank.merge(weather_wide, on='date', how='left')

print("paired shape:", paired.shape)
print(paired[['date', 'holiday_group', 'is_weekend', 'is_holiday', 'temp_00_04', 'relative_humidity_00_04', 'dewpoint_00_04']].head())

# 4. Aggregate weather station 066062 into time blocks 2004 - 2017

In [ ]:
# Parse datetime and strip timezone
weather['timestamp_utc'] = pd.to_datetime(weather['timestamp_utc'], utc=True)
weather = weather[
    (weather['timestamp_utc'].dt.year >= 2004) &
    (weather['timestamp_utc'].dt.year <= 2017)
]

weather['timestamp_local'] = weather['timestamp_utc'].dt.tz_localize(None)
weather['date'] = weather['timestamp_local'].dt.normalize()
weather['time_block'] = weather['timestamp_local'].dt.hour.map(lambda h:
    '00_04' if h < 4 else '04_10' if h < 10 else '10_15' if h < 15 else '15_20' if h < 20 else '20_24'
)

met_cols = ['temp', 'dewpoint', 'relative_humidity']
weather_agg = weather.groupby(['date', 'time_block'])[met_cols].mean().reset_index()

# Pivot to wide format: temp_00_04, dewpoint_10_15, etc.
weather_wide = weather_agg.pivot(index='date', columns='time_block', values=met_cols)
weather_wide.columns = [f"{var}_{block}" for var, block in weather_wide.columns]
weather_wide = weather_wide.reset_index()
weather_wide['date'] = pd.to_datetime(weather_wide['date']).dt.normalize()
weather_wide['weather_station_id'] = '066062'

print("weather_wide shape:", weather_wide.shape)
print(weather_wide.head())

# 5. Save weather reference csv
- new csv with temp, relative humidity and dewpoint means for the time blocks from 2004 - 2017 for the weather station 066062 (SYD OBS HILL)
- this can be used for substations that needed the combined before

In [ ]:
weather_ref_path = "/home/565/pv3484/aus_substation_electricity/data/BOM_NSW_weather_processed_v3/weather_station_066062_timeblock_means.csv"
weather_wide.to_csv(weather_ref_path, index=False)
print("Saved weather reference CSV:", weather_ref_path)

# 6. Merge demand rank with weather

In [ ]:
paired = mosman_rank.merge(
    weather_wide.drop(columns='weather_station_id'),
    on='date',
    how='left'
)

print("paired shape:", paired.shape)
print(paired[['date', 'holiday_group', 'is_weekend', 'is_holiday', 'temp_00_04', 'relative_humidity_00_04', 'dewpoint_00_04']].head(10))

# 5. Plot functions

In [ ]:
# --- Plot 1: Temperature (x) coloured by Dewpoint ---

def plot_temp_dewpoint_scatter(
    paired,
    holiday_name,
    station_code='MOSMA',
    station_full_name='Mosman',
    time_blocks=None
):
    if time_blocks is None:
        time_blocks = ['00_04', '04_10', '10_15', '15_20', '20_24']

    df = paired[paired['holiday_group'] == holiday_name].copy()
    if df.empty:
        print(f'No data for {holiday_name}')
        return None

    weekday_color = '#D8D8D8'
    weekend_color = 'skyblue'
    dp_cmap = plt.cm.RdYlGn
    block_titles = ['12am-4am', '4am-10am', '10am-3pm', '3pm-8pm', '8pm-12am']

    dp_cols_all = [f'dewpoint_{b}' for b in time_blocks if f'dewpoint_{b}' in df.columns]
    dp_global_min = df[dp_cols_all].min().min()
    dp_global_max = df[dp_cols_all].max().max()
    dp_norm = Normalize(vmin=dp_global_min, vmax=dp_global_max)

    fig, axes = plt.subplots(1, len(time_blocks), figsize=(4.5 * len(time_blocks), 5.0), sharey=False)

    for col_idx, block in enumerate(time_blocks):
        ax = axes[col_idx]
        rank_col = f'{block}_mean_relative_rank'
        x_col    = f'temp_{block}'
        dp_col   = f'dewpoint_{block}'

        ax.set_title(block_titles[col_idx], fontsize=14, pad=10)

        if rank_col not in df.columns or x_col not in df.columns or df[x_col].isna().all():
            ax.set_visible(False)
            continue

        wd = df[~df['is_weekend']]
        ax.scatter(wd[x_col], wd[rank_col], color=weekday_color, alpha=0.35, s=35, zorder=1)
        we = df[df['is_weekend']]
        ax.scatter(we[x_col], we[rank_col], color=weekend_color, alpha=0.45, s=35, zorder=1)

        holiday_rows = df[df['is_holiday']].copy()
        if dp_col in df.columns and not holiday_rows[dp_col].isna().all():
            colors = dp_cmap(dp_norm(holiday_rows[dp_col].values))
        else:
            colors = ['#AAAAAA'] * len(holiday_rows)

        ax.scatter(holiday_rows[x_col], holiday_rows[rank_col],
                   c=colors, s=120, edgecolor='black', linewidth=0.7, zorder=2)

        ax.set_xlabel('Temperature (°C)', fontsize=13)
        if col_idx == 0:
            ax.set_ylabel('Mean Relative Rank', fontsize=13)
        else:
            ax.set_ylabel('')
            ax.tick_params(axis='y', labelleft=False)
        ax.tick_params(axis='both', labelsize=10)
        ax.margins(0.05)

    fig.suptitle(
        f'{holiday_name} — {station_full_name}\n'
        f'Temperature vs Mean Relative Rank  |  Holiday coloured by Dewpoint',
        fontsize=16, y=1.05, linespacing=1.3
    )

    weekday_handle = plt.Line2D([], [], marker='o', linestyle='', color=weekday_color, markersize=8, label='Weekday')
    weekend_handle = plt.Line2D([], [], marker='o', linestyle='', color=weekend_color, markersize=8, label='Weekend')
    fig.legend(handles=[weekday_handle, weekend_handle],
               title='Day Type', title_fontsize=12, fontsize=12,
               loc='center', bbox_to_anchor=(0.33, 0.02),
               borderpad=0.4, labelspacing=0.3, handletextpad=0.4, ncol=1, frameon=True)

    fig.subplots_adjust(top=0.86, bottom=0.22, left=0.07, right=0.98, wspace=0.18)

    sm = ScalarMappable(cmap=dp_cmap, norm=dp_norm)
    sm.set_array([])
    cbar_ax = fig.add_axes([0.38, 0.04, 0.4, 0.04])
    cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal', extend='neither')
    cbar.set_label('Dewpoint (°C)', fontsize=12)
    cbar.ax.tick_params(labelsize=12)

    return fig

In [ ]:
# --- Plot 2: Relative Humidity (x) coloured by Year (tab20) ---

def plot_rh_year_scatter(
    paired,
    holiday_name,
    station_code='MOSMA',
    station_full_name='Mosman',
    time_blocks=None
):
    if time_blocks is None:
        time_blocks = ['00_04', '04_10', '10_15', '15_20', '20_24']

    df = paired[paired['holiday_group'] == holiday_name].copy()
    if df.empty:
        print(f'No data for {holiday_name}')
        return None

    df['year'] = pd.to_datetime(df['date']).dt.year
    years = sorted(df['year'].unique())
    year_palette = plt.cm.tab20.colors
    year_color_map = {yr: year_palette[i % len(year_palette)] for i, yr in enumerate(years)}

    weekday_color = '#D8D8D8'
    weekend_color = 'skyblue'
    block_titles = ['12am-4am', '4am-10am', '10am-3pm', '3pm-8pm', '8pm-12am']

    fig, axes = plt.subplots(1, len(time_blocks), figsize=(4.5 * len(time_blocks), 5.0), sharey=False)

    for col_idx, block in enumerate(time_blocks):
        ax = axes[col_idx]
        rank_col = f'{block}_mean_relative_rank'
        rh_col   = f'relative_humidity_{block}'

        ax.set_title(block_titles[col_idx], fontsize=14, pad=10)

        if rank_col not in df.columns or rh_col not in df.columns or df[rh_col].isna().all():
            ax.set_visible(False)
            continue

        wd = df[~df['is_weekend']]
        ax.scatter(wd[rh_col], wd[rank_col], color=weekday_color, alpha=0.35, s=35, zorder=1)
        we = df[df['is_weekend']]
        ax.scatter(we[rh_col], we[rank_col], color=weekend_color, alpha=0.45, s=35, zorder=1)

        holiday_rows = df[df['is_holiday']].copy()
        colors = [year_color_map[yr] for yr in holiday_rows['year']]
        ax.scatter(holiday_rows[rh_col], holiday_rows[rank_col],
                   c=colors, s=120, edgecolor='black', linewidth=0.7, zorder=2)

        ax.set_xlabel('Relative Humidity (%)', fontsize=13)
        if col_idx == 0:
            ax.set_ylabel('Mean Relative Rank', fontsize=13)
        else:
            ax.set_ylabel('')
            ax.tick_params(axis='y', labelleft=False)
        ax.tick_params(axis='both', labelsize=10)
        ax.margins(0.05)

    fig.suptitle(
        f'{holiday_name} — {station_full_name}\n'
        f'Relative Humidity vs Mean Relative Rank  |  Holiday coloured by Year',
        fontsize=16, y=1.05, linespacing=1.3
    )

    weekday_handle = plt.Line2D([], [], marker='o', linestyle='', color=weekday_color, markersize=8, label='Weekday')
    weekend_handle = plt.Line2D([], [], marker='o', linestyle='', color=weekend_color, markersize=8, label='Weekend')
    fig.legend(handles=[weekday_handle, weekend_handle],
               title='Day Type', title_fontsize=12, fontsize=12,
               loc='center', bbox_to_anchor=(0.33, 0.02),
               borderpad=0.4, labelspacing=0.3, handletextpad=0.4, ncol=1, frameon=True)

    year_handles = [mpatches.Patch(color=year_color_map[yr], label=str(yr)) for yr in years]
    fig.legend(handles=year_handles,
               title='Year', title_fontsize=12, fontsize=12,
               loc='center', bbox_to_anchor=(0.6, 0.02),
               borderpad=0.4, labelspacing=0.3, handletextpad=0.4,
               ncol=min(len(years), 7), frameon=True)

    fig.subplots_adjust(top=0.86, bottom=0.22, left=0.07, right=0.98, wspace=0.18)

    return fig

In [ ]:
# --- Plot 3: Relative Humidity (x) coloured by Temperature ---

def plot_rh_temp_scatter(
    paired,
    holiday_name,
    station_code='MOSMA',
    station_full_name='Mosman',
    time_blocks=None
):
    if time_blocks is None:
        time_blocks = ['00_04', '04_10', '10_15', '15_20', '20_24']

    df = paired[paired['holiday_group'] == holiday_name].copy()
    if df.empty:
        print(f'No data for {holiday_name}')
        return None

    weekday_color = '#D8D8D8'
    weekend_color = 'skyblue'
    temp_cmap = plt.cm.YlOrRd
    block_titles = ['12am-4am', '4am-10am', '10am-3pm', '3pm-8pm', '8pm-12am']

    temp_cols_all = [f'temp_{b}' for b in time_blocks if f'temp_{b}' in df.columns]
    temp_global_min = df[temp_cols_all].min().min()
    temp_global_max = df[temp_cols_all].max().max()
    temp_norm = Normalize(vmin=temp_global_min, vmax=temp_global_max)

    fig, axes = plt.subplots(1, len(time_blocks), figsize=(4.5 * len(time_blocks), 5.0), sharey=False)

    for col_idx, block in enumerate(time_blocks):
        ax = axes[col_idx]
        rank_col = f'{block}_mean_relative_rank'
        rh_col   = f'relative_humidity_{block}'
        temp_col = f'temp_{block}'

        ax.set_title(block_titles[col_idx], fontsize=14, pad=10)

        if rank_col not in df.columns or rh_col not in df.columns or df[rh_col].isna().all():
            ax.set_visible(False)
            continue

        wd = df[~df['is_weekend']]
        ax.scatter(wd[rh_col], wd[rank_col], color=weekday_color, alpha=0.35, s=35, zorder=1)
        we = df[df['is_weekend']]
        ax.scatter(we[rh_col], we[rank_col], color=weekend_color, alpha=0.45, s=35, zorder=1)

        holiday_rows = df[df['is_holiday']].copy()
        if temp_col in df.columns and not holiday_rows[temp_col].isna().all():
            colors = temp_cmap(temp_norm(holiday_rows[temp_col].values))
        else:
            colors = ['#AAAAAA'] * len(holiday_rows)

        ax.scatter(holiday_rows[rh_col], holiday_rows[rank_col],
                   c=colors, s=120, edgecolor='black', linewidth=0.7, zorder=2)

        ax.set_xlabel('Relative Humidity (%)', fontsize=13)
        if col_idx == 0:
            ax.set_ylabel('Mean Relative Rank', fontsize=13)
        else:
            ax.set_ylabel('')
            ax.tick_params(axis='y', labelleft=False)
        ax.tick_params(axis='both', labelsize=10)
        ax.margins(0.05)

    fig.suptitle(
        f'{holiday_name} — {station_full_name}\n'
        f'Relative Humidity vs Mean Relative Rank  |  Holiday coloured by Temperature',
        fontsize=16, y=1.05, linespacing=1.3
    )

    weekday_handle = plt.Line2D([], [], marker='o', linestyle='', color=weekday_color, markersize=8, label='Weekday')
    weekend_handle = plt.Line2D([], [], marker='o', linestyle='', color=weekend_color, markersize=8, label='Weekend')
    fig.legend(handles=[weekday_handle, weekend_handle],
               title='Day Type', title_fontsize=12, fontsize=12,
               loc='center', bbox_to_anchor=(0.33, 0.02),
               borderpad=0.4, labelspacing=0.3, handletextpad=0.4, ncol=1, frameon=True)

    fig.subplots_adjust(top=0.86, bottom=0.22, left=0.07, right=0.98, wspace=0.18)

    sm = ScalarMappable(cmap=temp_cmap, norm=temp_norm)
    sm.set_array([])
    cbar_ax = fig.add_axes([0.38, 0.04, 0.4, 0.04])
    cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal', extend='neither')
    cbar.set_label('Temperature (°C)', fontsize=12)
    cbar.ax.tick_params(labelsize=12)

    return fig

# 6. test plots

In [ ]:
test_holiday = "Australia Day"
print(f"Testing with: {test_holiday}")

fig1 = plot_temp_dewpoint_scatter(paired, test_holiday)
plt.show()

fig2 = plot_rh_year_scatter(paired, test_holiday)
plt.show()

fig3 = plot_rh_temp_scatter(paired, test_holiday)
plt.show()

# 7. save all

In [ ]:
save_dir = "/home/565/pv3484/aus_substation_electricity/data/figures/multi_dimension_mean_rank/seminar_mosma"

plot_types = {
    "temp_dewpoint": plot_temp_dewpoint_scatter,
    "rh_year":       plot_rh_year_scatter,
    "rh_temp":       plot_rh_temp_scatter,
}

for plot_type, plot_func in plot_types.items():
    folder = os.path.join(save_dir, plot_type)
    os.makedirs(folder, exist_ok=True)

    for holiday in paired["holiday_group"].dropna().unique():
        fig = plot_func(paired, holiday)
        if fig is not None:
            holiday_slug = holiday.replace(" ", "_").replace("'", "")
            save_path = os.path.join(folder, f"{holiday_slug}_{station_code}.png")
            fig.savefig(save_path, bbox_inches="tight", dpi=150)
            plt.close(fig)
            print(f"Saved: {plot_type}/{holiday_slug}")

print("Done.")